## NHD Network Analysis Demo
___TL;DR___: *We are trying to parallelize hydraulic calculations for dynamic subsets of the U.S. river and stream network*<br><br>
The following was developed as part of the process of preparing a method for forecasting flows on the US network of rivers and streams as represented in the National Hydrography Dataset (NHD). The NHD is a continuously evolving characterization of a fractal system so we felt that we needed to plan to have some flexibility. We hope to identify the complexity inherent in the network at different levels of resolution and we hope to be able to do so dynamically. The goal is also to be able to manage the complexity calculation for arbitrary collections of headwater points, such as might be obtained from a list of named streams or during a major flood event in a particular region.<br>
As a point of terminology, we use the word 'routing' as shorthand to refer to the computation of the translation of a particular flow condition, high or low, to downstream (or in some cases upstream) areas of influence.
The network complexity is related to the potential for parallelization of a serial analysis of the network. We have identified three levels of parallelization that may be implemented: 
1. System-level parallelization of independent networks -- the routing computations for the Mississippi River have little (nothing, except conceptual similarity and a shared existence on earth) to do with the computations for the Columbia river for any practical level of analysis. The system of networks across the US is what we are considering in general.
1. Network-level parallelization of interconnected reaches -- There is a need to consider the computations for adjacent branches within a network of con-flowing streams, but with proper ordering, some of the computations may be considered in parallel. For example, the Illinois River headwaters and the Mississippi River headwaters are related within their broader Mississippi network, but the routing calculations for those headwaters are pratically agnostic to one another.
1. Reach-level parallelization of the specific routing computation -- the numerical work of routing water downstream is a matrix computation and consists of exploring solutions to differential equations, all of which may potentially be examined in parallel, under the proper conditions and with suitable assumptions.<br>


### Import the package and prepare environment
This demo showcases routing network traversal and characterization using the `troute_network` package.
Ensure you have installed the package first: `pip install -e .` from the repository root.

In [ ]:
import sys
import subprocess

try:
    import google.colab
    ENV_IS_CL = True
    subprocess.run(["git", "clone", "https://github.com/NOAA-OWP/troute-network-analysis.git"])
    sys.path.append("/content/troute-network-analysis/src")
    subprocess.run(["pip", "install", "geopandas", "netcdf4"])
except:
    ENV_IS_CL = False

# default recursion limit (~1000) is slightly too small for the deepest branches of the network
sys.setrecursionlimit(6000)
# TODO: convert recursive functions to stack-based functions

### Some general functions and a test case
The next blocks use functions from the `troute_network` package (`nhd_network_utilities` and `recursive_print`) to create the `connections` objects which characterize the network to be analyzed.<br><br>
The `test_rows` object simulates a river network dataset. Each data row has a node ID, a 'to' node ID, and some other relevant data.

In [ ]:
try:
    import troute_network.nhd_network_utilities as nnu
    import troute_network.recursive_print as recursive_print
except ImportError:
    import nhd_network_utilities as nnu
    import recursive_print

if 1 == 1:
    """##TEST"""
    print("")
    print("Executing Test")
    # Test data
    test_rows = [
        [50, 178, 51, 0],
        [51, 178, 50, 0],
        [60, 178, 61, 0],
        [61, 178, 62, 0],
        [62, 178, 60, 0],
        [70, 178, 71, 0],
        [71, 178, 72, 0],
        [72, 178, 73, 0],
        [73, 178, 70, 0],
        [80, 178, 81, 0],
        [81, 178, 82, 0],
        [82, 178, 83, 0],
        [83, 178, 84, 0],
        [84, 178, 80, 0],
        [0, 456, -999, 0],
        [1, 178, 4, 0],
        [2, 394, 0, 0],
        [3, 301, 2, 0],
        [4, 798, 0, 0],
        [5, 679, 4, 0],
        [6, 523, 0, 0],
        [7, 815, 2, 0],
        [8, 841, -999, 0],
        [9, 514, 8, 0],
        [10, 458, 9, 0],
        [11, 832, 10, 0],
        [12, 543, 11, 0],
        [13, 240, 12, 0],
        [14, 548, 13, 0],
        [15, 920, 14, 0],
        [16, 920, 15, 0],
        [17, 514, 16, 0],
        [18, 458, 17, 0],
        [19, 832, 18, 0],
        [20, 543, 19, 0],
        [21, 240, 16, 0],
        [22, 548, 21, 0],
        [23, 920, 22, 0],
        [24, 240, 23, 0],
        [25, 548, 12, 0],
        [26, 920, 25, 0],
        [27, 920, 26, 0],
        [28, 920, 27, 0],
    ]

    test_key_col = 0
    test_downstream_col = 2
    test_length_col = 1
    test_terminal_code = -999
    debuglevel = 0
    verbose = True

    test_return_values = nnu.build_connections_object(
        geo_file_rows=test_rows,
        key_col=test_key_col,
        mask_set={row[test_key_col] for row in test_rows},
        downstream_col=test_downstream_col,
        length_col=test_length_col,
        terminal_code=test_terminal_code,
        verbose=verbose,
        debuglevel=debuglevel,
    )

    recursive_print.print_connections(
        headwater_keys=test_return_values[3],
        down_connections=test_return_values[0],
        up_connections=test_return_values[0],
        terminal_code=test_terminal_code,
        terminal_keys=test_return_values[4],
        terminal_ref_keys=test_return_values[5],
        debuglevel=debuglevel,
    )

    recursive_print.print_basic_network_info(
        connections=test_return_values[0],
        headwater_keys=test_return_values[3],
        junction_keys=test_return_values[7],
        terminal_keys=test_return_values[4],
        terminal_code=test_terminal_code,
        verbose=verbose,
        debuglevel=debuglevel,
    )

### Real Networks
Load actual river networks from shapefiles and netCDF.

In [ ]:
import os

if ENV_IS_CL:
    root = "/content/troute-network-analysis/"
else:
    root = os.path.dirname(os.path.abspath(""))
geo_input_folder = os.path.join(root, "test_data")

supernetworks = {}
"""##NHD Subset (Brazos/Lower Colorado)"""
supernetworks.update({"Brazos_LowerColorado_ge5": {"data": None, "values": None}})

# Check if the large RouteLink_CONUS netCDF file exists before adding the large networks
conus_nc_path = os.path.join(geo_input_folder, "Channels", "RouteLink_CONUS.nwm.v3.0.20.nc")
if os.path.exists(conus_nc_path):
    print("Found CONUS RouteLink file. Including CONUS datasets in demo.")
    """##NHD CONUS order 5 and greater"""
    supernetworks.update({"CONUS_ge5": {"data": None, "values": None}})
    """These are large -- be careful"""
    supernetworks.update({"CONUS_FULL_RES_v20": {"data": None, "values": None}})
else:
    print("CONUS RouteLink file not found. Skipping large CONUS networks in loop.")
    print(f"If you wish to test CONUS networks, place RouteLink_CONUS.nwm.v3.0.20.nc in {os.path.join(geo_input_folder, 'Channels')}")

debuglevel = -1
verbose = True

for sn in supernetworks:
    print(f"\nLoading supernetwork: {sn}")
    supernetworks[sn]["data"], supernetworks[sn]["values"] = nnu.set_networks(
        supernetwork=sn,
        geo_input_folder=geo_input_folder,
        verbose=verbose,
        debuglevel=debuglevel,
    )

In [ ]:
try:
    import troute_network.networkbuilder as nb
except ImportError:
    import networkbuilder as nb

if "CONUS_FULL_RES_v20" in supernetworks:
    terminal_keys = set([21676818])
    circular_keys = set([None])
    terminal_keys_super = terminal_keys - circular_keys
    con = supernetworks["CONUS_FULL_RES_v20"]["values"][0]
    terminal_code = supernetworks["CONUS_FULL_RES_v20"]["data"]["terminal_code"]
else:
    terminal_keys = supernetworks["Brazos_LowerColorado_ge5"]["values"][4]
    circular_keys = supernetworks["Brazos_LowerColorado_ge5"]["values"][6]
    terminal_keys_super = terminal_keys - circular_keys
    con = supernetworks["Brazos_LowerColorado_ge5"]["values"][0]
    terminal_code = supernetworks["Brazos_LowerColorado_ge5"]["data"]["terminal_code"]
    print(f"Using Brazos terminal keys: {terminal_keys_super}")

### With this, we can separate the different rivers in the network
Once a 'connection' object has been created with a representation of the river network, we can traverse that object and perform calculations -- in the example below, we parallelize the process of traversing the independent portions of the network and then serially compute the number of junctions. This corresponds to the "**System-level parallelization**" mentioned as _item 1_ above.
### NOW for the next step
We could compute total upstream length or flow due to incoming lateral contributions accumulated over the entire upstream network. That second calculation can also be parallelized but we have to figure out how to accomplish it intelligently so that the collective calculation is network-aware.

In [ ]:
# parallel compute
import time
import multiprocessing
from functools import partial

if "CONUS_FULL_RES_v20" in supernetworks:
    print("Profiling with CONUS Full Res network")
    terminal_keys = supernetworks["CONUS_FULL_RES_v20"]["values"][4]
    terminal_keys = terminal_keys.union(set([21676818, 22274808, 21661814]))
    circular_keys = supernetworks["CONUS_FULL_RES_v20"]["values"][6]
    terminal_keys_super = terminal_keys - circular_keys
    con = supernetworks["CONUS_FULL_RES_v20"]["values"][0]
    terminal_code = supernetworks["CONUS_FULL_RES_v20"]["data"]["terminal_code"]
else:
    print("Profiling with Brazos & Lower Colorado ge5 network")
    terminal_keys = supernetworks['Brazos_LowerColorado_ge5']['values'][4]
    circular_keys = supernetworks['Brazos_LowerColorado_ge5']['values'][6]
    terminal_keys_super = terminal_keys - circular_keys
    con = supernetworks['Brazos_LowerColorado_ge5']['values'][0]
    terminal_code = supernetworks['Brazos_LowerColorado_ge5']['data']['terminal_code']

In [ ]:
def recursive_junction_read(
    keys, network, terminal_code=0, verbose=False, debuglevel=0
):
    global con
    for key in keys:
        ckey = key
        try:
            ukeys = con[key]["upstreams"]
            while not len(ukeys) >= 2 and not (ukeys == {terminal_code}):
                if debuglevel <= -3:
                    print(f"segs at ckey {ckey}: {network['segment_count']}")
                # the terminal code will indicate a headwater
                if debuglevel <= -4:
                    print(ukeys)
                (ckey,) = ukeys
                ukeys = con[ckey]["upstreams"]
            if ukeys == {terminal_code}:
                if debuglevel <= -3:
                    print(f"headwater found at {ckey}")
                network["segment_count"] += 1
                if debuglevel <= -3:
                    print(f"segs at ckey {ckey}: {network['segment_count']}")
            elif len(ukeys) >= 2:
                network["segment_count"] += 1
                if debuglevel <= -3:
                    print(f"junction found at {ckey} with upstreams {ukeys}")
                network["segment_count"] += 1
                if debuglevel <= -3:
                    print(f"segs at ckey {ckey}: {network['segment_count']}")
                network["junction_count"] += 1  # the Terminal Segment
                recursive_junction_read(
                    ukeys,
                    network,
                    terminal_code=terminal_code,
                    verbose=verbose,
                    debuglevel=debuglevel,
                )
                # print(ukeys)
                ukeys = con[ckey]["upstreams"]
                ckey = ukeys
        except:
            if debuglevel <= -2:
                print(f"There is a problem with connection: {key}: {con[key]}")


def network_trace(nid, terminal_code=terminal_code, verbose=False, debuglevel=0):

    network = {}
    global con
    us_length_total = 0

    if verbose:
        print(f"\ntraversing upstream on network {nid}:")
    # try:
    if 1 == 1:
        network.update({"junction_count": 0})
        network.update({"segment_count": 0})  # the Terminal Segment
        recursive_junction_read(
            [nid],
            network,
            verbose=verbose,
            terminal_code=terminal_code,
            debuglevel=debuglevel,
        )
        if verbose:
            print(f"junctions: {network['junction_count']}")
        if verbose:
            print(f"segments: {network['segment_count']}")
    # except Exception as exc:
    #     print(exc)
    # TODO: compute upstream length as a surrogate for the routing computation
    return {nid: network, "upstream_length": us_length_total}

In [ ]:
###continuing from previous cell
if "CONUS_FULL_RES_v20" in supernetworks:
    networks = {
        terminal_key: {}
        for terminal_key in terminal_keys_super
        if terminal_key in [
            21661814, # Cahaba river ID
            22274808, # Walnut creek ID
            21676818, # Mulberry creek
        ]
    }
else:
    # Profile using a subset of terminal networks in Brazos Basin
    networks = {
        terminal_key: {}
        for terminal_key in sorted(list(terminal_keys_super))[:5]
    }

debuglevel = -3
verbose = True

if verbose:
    print("verbose output")
if verbose:
    print(f"number of Independent Networks to be analyzed is {len(networks)}")
if verbose:
    print(f"Multi-processing will use {multiprocessing.cpu_count()} CPUs")
if verbose:
    print(f"debuglevel is {debuglevel}")

start_time = time.time()
results_serial = {}
for nid, network in networks.items():
    network.update(
        network_trace(
            nid, terminal_code=terminal_code, verbose=verbose, debuglevel=debuglevel
        )[nid]
    )
print("--- %s seconds: serial compute ---" % (time.time() - start_time))
if debuglevel <= -1:
    print(len(networks.items()))
if debuglevel <= -2:
    print(networks)

In [ ]:
## Notice that I'm not timing the initialization in each case,
## which might be considered cheating a little bit.
## I timed it for the first case for reference.
nids = (nid for nid in networks)
start_time = time.time()
with multiprocessing.Pool() as pool:
    print(
        "--- %s seconds: parallel overhead to load multiprocessing.Pool() ---"
        % (time.time() - start_time)
    )
    start_time = time.time()
    results = pool.map(network_trace, nids)
    print(
        "--- %s seconds: parallel compute using default terminal_code ---"
        % (time.time() - start_time)
    )
if debuglevel <= -1:
    print(len(results))
if debuglevel <= -2:
    print(results)

nids = (nid for nid in networks)
snt = partial(network_trace, terminal_code=terminal_code)
with multiprocessing.Pool() as pool:
    start_time = time.time()
    results = pool.map(snt, nids)
    print(
        "--- %s seconds: parallel compute using partial function ---"
        % (time.time() - start_time)
    )
if debuglevel <= -1:
    print(len(results))
if debuglevel <= -2:
    print(results)

nidsWtc = ([nid, terminal_code] for nid in networks)
with multiprocessing.Pool() as pool:
    start_time = time.time()
    results = pool.starmap(network_trace, nidsWtc)
    print(
        "--- %s seconds: parallel compute with list of lists and starmap ---"
        % (time.time() - start_time)
    )
if debuglevel <= -1:
    print(len(results))
if debuglevel <= -2:
    print(results)

### Colab output

For the 14351 independent networks (and 2.7M segments) of the NWM Full Resolution dataset on a Google colaboratory VM, the algorithm steps through the segments of the different independent networks in order in about 4 seconds, regardless of the parallelization method. On a modest workstation at NWC with 12 cores, the compute time was approximately 2 seconds for a serial compute and 1.1 seconds for each of the parallel methods. Notably, because we are only breaking apart the independent networks, the maximum parallel speedup is limited by the size of the largest network -- the Mississippi River. By executing with the terminal node for the Mississippi River removed from the evaluation set, the serial calculation takes only 1.4 seconds and the parallel executions drop to approximately 0.4 seconds.
